In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/sanidhyavijay24/adaptive-blending-auc-0-95410/__results__.html
/kaggle/input/notebooks/sanidhyavijay24/adaptive-blending-auc-0-95410/submission.csv
/kaggle/input/notebooks/sanidhyavijay24/adaptive-blending-auc-0-95410/__notebook__.ipynb
/kaggle/input/notebooks/sanidhyavijay24/adaptive-blending-auc-0-95410/__output__.json
/kaggle/input/notebooks/sanidhyavijay24/adaptive-blending-auc-0-95410/custom.css
/kaggle/input/notebooks/azzamradman/submit-and-hold-onto-your-wig/__results__.html
/kaggle/input/notebooks/azzamradman/submit-and-hold-onto-your-wig/submission.csv
/kaggle/input/notebooks/azzamradman/submit-and-hold-onto-your-wig/__notebook__.ipynb
/kaggle/input/notebooks/azzamradman/submit-and-hold-onto-your-wig/__output__.json
/kaggle/input/notebooks/azzamradman/submit-and-hold-onto-your-wig/custom.css
/kaggle/input/notebooks/azzamradman/submit-and-hold-onto-your-wig/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/artemevstafyev/shap-analysis-

In [2]:
"""
Heart Disease Prediction - Optimized Median Blend
==================================================

Building on the success of median (0.95397), create enhanced versions.

Target: 0.9540+ AUC
"""

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("Optimized Median Blend - Target 0.9540+")
print("="*70)

# ==========================================
# LOAD SUBMISSIONS
# ==========================================
print("\nLoading submissions...")

submissions = []

paths = [
    '/kaggle/input/notebooks/azzamradman/submit-and-hold-onto-your-wig/submission.csv',
    '/kaggle/input/notebooks/artemevstafyev/shap-analysis-high-cv-score/submission.csv',
    '/kaggle/input/notebooks/dmahajanbe23/predicting-heart-disease-auc-0-95410/submission.csv',
    '/kaggle/input/notebooks/itasps/predicting-heart-disease-ensemble/submission.csv',
    '/kaggle/input/notebooks/sanidhyavijay24/adaptive-blending-auc-0-95410/submission.csv',
    '/kaggle/input/notebooks/yusufmurtaza01/s6e2-blend/submission_t2.csv',
    '/kaggle/input/notebooks/yusufmurtaza01/s6e2-blend/submission_bokeh.csv',
    '/kaggle/input/notebooks/yusufmurtaza01/s6e2-blend/submission_top4.csv',
    '/kaggle/input/notebooks/yusufmurtaza01/s6e2-blend3/submission_t2.csv',
    '/kaggle/input/notebooks/yusufmurtaza01/s6e2-blend3/submission.csv',
]

for path in paths:
    try:
        sub = pd.read_csv(path)
        submissions.append(sub.iloc[:, 1].values)
        print(f"✓ {path.split('/')[-1]}")
    except:
        print(f"✗ {path.split('/')[-1]}")

print(f"\nTotal loaded: {len(submissions)}")

df = pd.DataFrame({f's{i}': s for i, s in enumerate(submissions)})

# ==========================================
# MEDIAN-BASED STRATEGIES
# ==========================================
print("\n" + "="*70)
print("MEDIAN-BASED STRATEGIES")
print("="*70)

# 1. Simple Median (baseline - 0.95397)
blend1 = df.median(axis=1).values
print("✓ Strategy 1: Simple Median")

# 2. Trimmed Median (remove extreme 5% on each end)
def trimmed_median(row, trim=0.05):
    sorted_vals = np.sort(row)
    n = len(sorted_vals)
    start = int(n * trim)
    end = int(n * (1 - trim))
    return np.median(sorted_vals[start:end])

blend2 = np.apply_along_axis(trimmed_median, 1, df.values)
print("✓ Strategy 2: Trimmed Median (5%)")

# 3. Weighted Median (favor submissions with less extreme values)
def weighted_median(row):
    # Weight by inverse of distance from row median
    row_median = np.median(row)
    distances = np.abs(row - row_median)
    weights = 1 / (distances + 0.001)
    weights = weights / weights.sum()

    sorted_idx = np.argsort(row)
    sorted_weights = weights[sorted_idx]
    cumsum = np.cumsum(sorted_weights)

    median_idx = np.searchsorted(cumsum, 0.5)
    return row[sorted_idx[median_idx]]

blend3 = np.apply_along_axis(weighted_median, 1, df.values)
print("✓ Strategy 3: Weighted Median")

# 4. Median + Mean hybrid
blend4 = 0.65 * df.median(axis=1).values + 0.35 * df.mean(axis=1).values
print("✓ Strategy 4: Median-Mean Hybrid (65/35)")

# 5. Median of ranked predictions
ranked = np.apply_along_axis(
    lambda x: pd.Series(x).rank(method='average').to_numpy(), 0, df
)
ranked_df = pd.DataFrame(ranked)
blend5_ranks = ranked_df.median(axis=1).values
blend5 = (blend5_ranks - blend5_ranks.min()) / (blend5_ranks.max() - blend5_ranks.min())
print("✓ Strategy 5: Rank Median")

# ==========================================
# ENSEMBLE ALL MEDIAN STRATEGIES
# ==========================================
print("\n" + "="*70)
print("FINAL ENSEMBLE")
print("="*70)

# Combine all median strategies
final_pred = (
    0.30 * blend1 +  # Simple median (proven 0.95397)
    0.25 * blend2 +  # Trimmed median
    0.20 * blend4 +  # Median-mean hybrid
    0.15 * blend5 +  # Rank median
    0.10 * blend3    # Weighted median
)

final_pred = np.clip(final_pred, 0, 1)

print("\nWeights:")
print("  30% - Simple Median (proven 0.95397)")
print("  25% - Trimmed Median")
print("  20% - Median-Mean Hybrid")
print("  15% - Rank Median")
print("  10% - Weighted Median")

# ==========================================
# CREATE SUBMISSION
# ==========================================
sample = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')
sample['Heart Disease'] = final_pred
sample.to_csv('submission.csv', index=False)

print("\n" + "="*70)
print("✓ OPTIMIZED MEDIAN BLEND COMPLETE")
print("="*70)

print(f"\nPrediction Stats:")
print(f"  Min:    {final_pred.min():.5f}")
print(f"  Max:    {final_pred.max():.5f}")
print(f"  Mean:   {final_pred.mean():.5f}")
print(f"  Median: {np.median(final_pred):.5f}")

print(f"\nBlended {len(submissions)} submissions")
print("Using 5 median-based strategies")
print("\nPrevious best: 0.95397 (simple median)")
print("Expected: 0.9540+ (enhanced median strategies)")
print("="*70)

print("\nFirst 10 predictions:")
print(sample.head(10))

# Also save individual strategies for comparison
sample['Heart Disease'] = blend2
sample.to_csv('submission_trimmed_median.csv', index=False)

sample['Heart Disease'] = blend4
sample.to_csv('submission_hybrid.csv', index=False)

print("\nAlternative files created:")
print("  submission_trimmed_median.csv")
print("  submission_hybrid.csv")

Optimized Median Blend - Target 0.9540+

Loading submissions...
✓ submission.csv
✓ submission.csv
✓ submission.csv
✓ submission.csv
✓ submission.csv
✓ submission_t2.csv
✓ submission_bokeh.csv
✓ submission_top4.csv
✓ submission_t2.csv
✓ submission.csv

Total loaded: 10

MEDIAN-BASED STRATEGIES
✓ Strategy 1: Simple Median
✓ Strategy 2: Trimmed Median (5%)
✓ Strategy 3: Weighted Median
✓ Strategy 4: Median-Mean Hybrid (65/35)
✓ Strategy 5: Rank Median

FINAL ENSEMBLE

Weights:
  30% - Simple Median (proven 0.95397)
  25% - Trimmed Median
  20% - Median-Mean Hybrid
  15% - Rank Median
  10% - Weighted Median

✓ OPTIMIZED MEDIAN BLEND COMPLETE

Prediction Stats:
  Min:    0.00000
  Max:    0.99933
  Mean:   0.45920
  Median: 0.35065

Blended 10 submissions
Using 5 median-based strategies

Previous best: 0.95397 (simple median)
Expected: 0.9540+ (enhanced median strategies)

First 10 predictions:
       id  Heart Disease
0  630000       0.923152
1  630001       0.021066
2  630002       0.975